# A2 RNN's full model

# Function definitions for creating the datasets

First we need to create our datasets that are going to be used for training our models.

In order to create image queries of simple arithmetic operations such as '15+13' or '42-10' we need to create images of '+' and '-' signs using ***open-cv*** library. We will use these operand signs together with the MNIST dataset to represent the digits.

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import Dense, RNN, LSTM, Flatten, TimeDistributed, LSTMCell
from tensorflow.keras.layers import RepeatVector, Conv2D, SimpleRNN, GRU, Reshape, ConvLSTM2D, Conv2DTranspose

2026-01-06 12:07:06.867505: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-06 12:07:06.900046: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-06 12:07:15.817646: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from scipy.ndimage import rotate


# Create plus/minus operand signs
def generate_images(number_of_images=50, sign='-'):
    blank_images = np.zeros([number_of_images, 28, 28])  # Dimensionality matches the size of MNIST images (28x28)
    x = np.random.randint(12, 16, (number_of_images, 2)) # Randomized x coordinates
    y1 = np.random.randint(6, 10, number_of_images)       # Randomized y coordinates
    y2 = np.random.randint(18, 22, number_of_images)     # -||-

    for i in range(number_of_images): # Generate n different images
        cv2.line(blank_images[i], (y1[i], x[i,0]), (y2[i], x[i, 1]), (255,0,0), 2, cv2.LINE_AA)     # Draw lines with randomized coordinates
        if sign == '+':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA) # Draw lines with randomized coordinates

    return blank_images

def show_generated(images, n=5):
    plt.figure(figsize=(2, 2))
    for i in range(n**2):
        plt.subplot(n, n, i+1)
        plt.axis('off')
        plt.imshow(images[i])
    plt.show()

In [3]:
def create_data(highest_integer, num_addends=2, operands=['+', '-']):
    """
    Creates the following data for all pairs of integers up to [1:highest integer][+/-][1:highest_integer]:

    @return:
    X_text: '51+21' -> text query of an arithmetic operation (5)
    X_img : Stack of MNIST images corresponding to the query (5 x 28 x 28) -> sequence of 5 images of size 28x28
    y_text: '72' -> answer of the arithmetic text query
    y_img :  Stack of MNIST images corresponding to the answer (3 x 28 x 28)

    Images for digits are picked randomly from the whole MNIST dataset.
    """

    num_indices = [np.where(MNIST_labels==x) for x in range(10)]
    num_data = [MNIST_data[inds] for inds in num_indices]
    image_mapping = dict(zip(unique_characters[:10], num_data))
    image_mapping['-'] = generate_images()
    image_mapping['+'] = generate_images(sign='+')
    image_mapping['*'] = generate_images(sign='*')
    image_mapping[' '] = np.zeros([1, 28, 28])

    X_text, X_img, y_text, y_img = [], [], [], []

    for i in range(highest_integer + 1):      # First addend
        for j in range(highest_integer + 1):  # Second addend
            for sign in operands: # Create all possible combinations of operands
                query_string = to_padded_chars(str(i) + sign + str(j), max_len=max_query_length, pad_right=True)
                query_image = []
                for n, char in enumerate(query_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    query_image.append(image_set[index].squeeze())

                result = eval(query_string)
                result_string = to_padded_chars(result, max_len=max_answer_length, pad_right=True)
                result_image = []
                for n, char in enumerate(result_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    result_image.append(image_set[index].squeeze())

                X_text.append(query_string)
                X_img.append(np.stack(query_image))
                y_text.append(result_string)
                y_img.append(np.stack(result_image))

    return np.stack(X_text), np.stack(X_img)/255., np.stack(y_text), np.stack(y_img)/255.

def to_padded_chars(integer, max_len=3, pad_right=False):
    """
    Returns a string of len()=max_len, containing the integer padded with ' ' on either right or left side
    """
    length = len(str(integer))
    padding = (max_len - length) * ' '
    if pad_right:
        return str(integer) + padding
    else:
        return padding + str(integer)


# Creating our data

The dataset consists of 20000 samples that (additions and subtractions between all 2-digit integers) and they have two kinds of inputs and label modalities:

  **X_text**: strings containing queries of length 5: ['  1+1  ', '11-18', ...]

  **X_image**: a stack of images representing a single query, dimensions: [5, 28, 28]

  **y_text**: strings containing answers of length 3: ['  2', '156']

  **y_image**: a stack of images that represents the answer to a query, dimensions: [3, 28, 28]

In [4]:
# Illustrate the generated query/answer pairs

unique_characters = '0123456789+- '       # All unique characters that are used in the queries (13 in total: digits 0-9, 2 operands [+, -], and a space character ' '.)
highest_integer = 99                      # Highest value of integers contained in the queries

max_int_length = len(str(highest_integer))# Maximum number of characters in an integer
max_query_length = max_int_length * 2 + 1 # Maximum length of the query string (consists of two integers and an operand [e.g. '22+10'])
max_answer_length = 3    # Maximum length of the answer string (the longest resulting query string is ' 1-99'='-98')

# Create the data (might take around a minute)
(MNIST_data, MNIST_labels), _ = tf.keras.datasets.mnist.load_data()
X_text, X_img, y_text, y_img = create_data(highest_integer)
print(X_text.shape, X_img.shape, y_text.shape, y_img.shape)




(20000,) (20000, 5, 28, 28) (20000,) (20000, 3, 28, 28)


# My own helper functions

In the models below teacher forcing is used. For this the vocabulary will need a start and end token. Subsequently the one-hot encoding and decoding functions need to be altered to include these.

In [5]:
# Teacher forcing preparation
vocabulary_tf = list(unique_characters)+['<start>','<end>'] 

indices = {char:i for i, char in enumerate(vocabulary_tf)}
reverse_indices={i:char for i,char in enumerate(vocabulary_tf)}


# One-hot encoding
def encode_labels_tf(labels, vocabulary=vocabulary_tf, indices_map=indices):
  n = len(labels)
  length = len(labels[0])+2 # for start of sequence <start> and end of sequence <end> tokens.

  one_hot = np.zeros([n, length, len(vocabulary)])
  for i, label in enumerate(labels):
    full_label = ['<start>'] + list(label) + ['<end>']
    m = np.zeros([length, len(vocabulary)])
    for j, char in enumerate(full_label):
       m[j, indices_map[char]] = 1
    one_hot[i] = m

  return one_hot

# One-hot decoding
def decode_labels_tf(labels, indices_map=reverse_indices):

    pred_indices = np.argmax(labels, axis=-1) 
    
    decoded_list = []
    for sequence in pred_indices:

        chars = [indices_map[i] for i in sequence if indices_map[i] not in ['<start>', '<end>', '<pad>']]
        decoded_list.append(''.join(chars))
        
    return decoded_list

X_text_onehot = encode_labels_tf(X_text)
y_text_onehot = encode_labels_tf(y_text)

print(X_text_onehot.shape, y_text_onehot.shape)

(20000, 7, 15) (20000, 5, 15)


## Creating data

In [ ]:
# Creating visual encoder training data, including ground-truth sequences used for teacher forcing.
size=0.1

X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X_img, X_text_onehot, random_state=42, test_size=size
)

X_train_pt, X_val_pt, y_train_pt, y_val_pt = train_test_split(
    X_train_pt, y_train_pt, random_state=42, test_size=size/(1-size)
)

max_answer_length_pt = 6

y_train_in_pt = y_train_pt[:, :-1, :]
y_train_target_pt = y_train_pt[:, 1:, :]

y_val_in_pt = y_val_pt[:, :-1, :]
y_val_target_pt = y_val_pt[:, 1:, :]

y_test_in_pt = y_test_pt[:, :-1, :]
y_test_target_pt = y_test_pt[:, 1:, :]

In [ ]:
# Calculator model data
size=0.1

X_train_calc, X_test_calc, y_train_calc, y_test_calc = train_test_split(
    X_text_onehot[:, :-1, :], y_text_onehot, random_state=42, test_size=size
)

X_train_calc, X_val_calc, y_train_calc, y_val_calc = train_test_split(
    X_train_calc, y_train_calc, random_state=42, test_size=size/(1-size)
)

max_answer_length_tf = 4

y_train_in_calc = y_train_calc[:, :-1, :]
y_train_target_calc = y_train_calc[:, 1:, :]

y_val_in_calc = y_val_calc[:, :-1, :]
y_val_target_calc = y_val_calc[:, 1:, :]

y_test_in_calc = y_test_calc[:, :-1, :]
y_test_target_calc = y_test_calc[:, 1:, :]

In [ ]:
# Full pipeline training data
size=0.1

X_train, X_test, y_train, y_test = train_test_split(
    X_img, y_text_onehot, random_state=42, test_size=size
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, random_state=42, test_size=size/(1-size)
)

y_train_in = y_train[:, :-1, :]
y_train_target = y_train[:, 1:, :]

y_val_in = y_val[:, :-1, :]
y_val_target = y_val[:, 1:, :]

y_test_in = y_test[:, :-1, :]
y_test_target = y_test[:, 1:, :]

## Full model

In [66]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Lambda
from tensorflow.keras.models import Model

def build_image2text_no_encoder_forcing(pretraining_model, calculator_model):
    # Inputs
    X_in = Input(shape=(5, 28, 28, 1), name='image_sequence')
    
    # We still keep expr_in in the input list so your data pipeline doesn't break,
    # but we will effectively "mute" it before it reaches the pretraining_model.
    expr_in = Input(shape=(6, 15), name='expression_input_masked') 
    
    ans_in = Input(shape=(4, 15), name='answer_teacher_forcing')

    # 1. Create a Zero-Mask for the Encoder
    # This ensures the pretraining_model only sees the image sequence.
    # Academically, this is "Ablating" the teacher signal from the first stage.
    masked_expr = Lambda(lambda x: tf.zeros_like(x))(expr_in)

    # 2. Visual Encoder (Now only using pixels for features)
    expression_probs = pretraining_model([X_in, masked_expr])

    # 3. Calculator with Teacher Forcing (Kept for training stability)
    final_answer_probs = calculator_model([expression_probs, ans_in])

    full_pipeline = Model(
        inputs=[X_in, expr_in, ans_in], 
        outputs=final_answer_probs, 
        name="partial_teacher_forcing_model"
    )
    
    return full_pipeline

In [ ]:
#old
from tensorflow.keras.models import Model

def build_image2text(pretraining_model, calculator_model):
    # Inputs
    X_in = Input(shape=(5, 28, 28, 1), name='image_sequence')
    expr_in = Input(shape=(6, 15), name='expression_teacher_forcing') # For Encoder
    ans_in = Input(shape=(4, 15), name='answer_teacher_forcing')     # For Calculator

    # 1. Visual Encoder with Teacher Forcing
    expression_probs = pretraining_model([X_in, expr_in])

    # 2. Calculator with Teacher Forcing
    final_answer_probs = calculator_model([expression_probs, ans_in])

    full_pipeline = Model(inputs=[X_in, expr_in, ans_in], 
                          outputs=final_answer_probs, 
                          name="dual_teacher_forcing_model")
    
    return full_pipeline

### Training models

#### Visual encoder

In [30]:
# Training the visual encoder. We train this beforehand.
## Building model
dropout=0.5
RLstrength=0
max_size=256 # Must be 128 at minimum. Must be an even number.
image2text_pretraining = build_image2text_pretraining(dropout=dropout, RLstrength=RLstrength, max_size=max_size)


## Compile
learning_rate=4.0e-4 # Initial LR
weight_decay=1.0e-2  # Decoupled Weight Decay

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

image2text_pretraining.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])


## Training
stopper_patience_warmup = 5

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience_warmup,
    restore_best_weights=True
)

### Warm-start
history = image2text_pretraining.fit(x=[X_train_pt, y_train_in_pt], y=y_train_target_pt, 
               epochs = 50,
               batch_size = 32,
               validation_data = ([X_val_pt, y_val_in_pt], y_val_target_pt),
               callbacks = [early_stopper])


Model: "pretraining_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_model       │ [(None, 5, 7, 7,  │    906,560 │ sequence[0][0]    │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ input_layer_23 │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_1… │ 1)                │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        320 │ -                 │
│ time_distributed_1… │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        128 │ -                 │
│ time_distributed_1… │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_1… │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_1… │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_1… │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │     18,496 │ -                 │
│ time_distributed_1… │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │        256 │ -                 │
│ time_distributed_1… │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_1… │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_1… │ 64)               │            │                 

 Total params: 1,254,991 (4.79 MB)

 Trainable params: 1,254,799 (4.79 MB)

 Non-trainable params: 192 (768.00 B)

...

E0000 00:00:1767624910.498147 1163468 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/convolution_6' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_6', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_

500/500 ━━━━━━━━━━━━━━━━━━━━ 27s 46ms/step - categorical_accuracy: 0.4112 - loss: 1.8407 - val_categorical_accuracy: 0.4230 - val_loss: 1.8753
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 23s 45ms/step - categorical_accuracy: 0.5579 - loss: 1.5248 - val_categorical_accuracy: 0.4494 - val_loss: 1.9434
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - categorical_accuracy: 0.7162 - loss: 1.2292 - val_categorical_accuracy: 0.5359 - val_loss: 1.7117
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - categorical_accuracy: 0.8322 - loss: 1.0079 - val_categorical_accuracy: 0.6814 - val_loss: 1.3148
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - categorical_accuracy: 0.8960 - loss: 0.8720 - val_categorical_accuracy: 0.7210 - val_loss: 1.2504
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - categorical_accuracy: 0.9258 - loss: 0.7963 - val_categorical_accuracy: 0.7411 - val_loss: 1.1924
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - categorical_accuracy: 0.9419 - 

E0000 00:00:1767625557.859276 1163468 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/convolution_6' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_6', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_

500/500 ━━━━━━━━━━━━━━━━━━━━ 23s 39ms/step - categorical_accuracy: 0.9859 - loss: 0.6008 - val_categorical_accuracy: 0.9709 - val_loss: 0.6425 - learning_rate: 1.0000e-05
Epoch 2/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.9865 - loss: 0.5988 - val_categorical_accuracy: 0.9709 - val_loss: 0.6420 - learning_rate: 1.0000e-05
Epoch 3/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.9871 - loss: 0.5972 - val_categorical_accuracy: 0.9696 - val_loss: 0.6470 - learning_rate: 1.0000e-05
Epoch 4/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.9877 - loss: 0.5967 - val_categorical_accuracy: 0.9718 - val_loss: 0.6414 - learning_rate: 1.0000e-05
Epoch 5/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.9872 - loss: 0.5970 - val_categorical_accuracy: 0.9733 - val_loss: 0.6381 - learning_rate: 1.0000e-05
Epoch 6/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.9873 - loss: 0.5967 -

KeyboardInterrupt: 

In [31]:
## Recompile so AdamW moments are reset

learning_rate=1.0e-5
weight_decay=5.0e-2
optimizer = tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay)
image2text_pretraining.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])


scheduler_patience = 5
stopper_patience = 20

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

### Fine-tune training
history = image2text_pretraining.fit(x=[X_train_pt, y_train_in_pt], y=y_train_target_pt, 
               epochs = 60,
               batch_size = 32,
               validation_data = ([X_val_pt, y_val_in_pt], y_val_target_pt),
               callbacks=[lr_scheduler, early_stopper],
               verbose=1)


Epoch 1/60


E0000 00:00:1767626003.428553 1163468 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_90/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_343/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/gradients/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Tanh_1_grad/TanhGrad' -> 'StatefulPartitionedCall/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_343/gra

500/500 ━━━━━━━━━━━━━━━━━━━━ 26s 46ms/step - categorical_accuracy: 0.9884 - loss: 0.5937 - val_categorical_accuracy: 0.9696 - val_loss: 0.6481 - learning_rate: 1.0000e-05
Epoch 2/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - categorical_accuracy: 0.9879 - loss: 0.5941 - val_categorical_accuracy: 0.9697 - val_loss: 0.6497 - learning_rate: 1.0000e-05
Epoch 3/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - categorical_accuracy: 0.9882 - loss: 0.5939 - val_categorical_accuracy: 0.9697 - val_loss: 0.6490 - learning_rate: 1.0000e-05
Epoch 4/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - categorical_accuracy: 0.9883 - loss: 0.5941 - val_categorical_accuracy: 0.9678 - val_loss: 0.6540 - learning_rate: 1.0000e-05
Epoch 5/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - categorical_accuracy: 0.9885 - loss: 0.5928 - val_categorical_accuracy: 0.9682 - val_loss: 0.6534 - learning_rate: 1.0000e-05
Epoch 6/60
499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - categorical_accuracy: 0.9882 - loss: 0.5933
Ep

#### Calculator

In [32]:
# Training the calculator

## Building model
dropout=0.5
RLstrength=0
text2text_calculator = build_text2text_calc(dropout=dropout, RLstrength=RLstrength, max_size=max_size)

## Compile
learning_rate=5.0e-4 
weight_decay=1.0e-4  

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

text2text_calculator.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])

## Training
stopper_patience_warmup = 5

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience_warmup,
    restore_best_weights=True
)

### Warm-start
history_warmup = text2text_calculator.fit(x=[X_train_calc, y_train_in_calc], y=y_train_target_calc, 
               epochs = 50,
               batch_size = 32,
               validation_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
               callbacks = [early_stopper])

Model: "calculator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ expression_input    │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ answer (InputLayer) │ (None, 4, 15)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ calculator_encoder  │ [(None, 6, 256),  │    278,528 │ expression_input… │
│ (LSTM)              │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 4, 256),  │    278,528 │ answer[0][0],     │
│                     │ (None, 256),      │            │ calculator_encod… │
│                     │ (None, 256)]      │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_block     │ (None, 4, 256)    │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_q_A          │ (None, 4, 512)    │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_block[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_dense      │ (None, 4, 256)    │    131,328 │ concat_q_A[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_res_dense   │ (None, 4, 256)    │      3,840 │ answer[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_with_resid… │ (None, 4, 256)    │          0 │ combined_dense[0… │
│ (Add)               │                   │            │ decoder_res_dens… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_activation  │ (None, 4, 256)    │          0 │ decoder_with_res… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layer_norm  │ (None, 4, 256)    │        512 │ decoder_activati… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 4, 15)     │      3,855 │ decoder_layer_no… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 696,591 (2.66 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - categorical_accuracy: 0.5404 - loss: 1.9385 - val_categorical_accuracy: 0.5665 - val_loss: 1.7485
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - categorical_accuracy: 0.5868 - loss: 1.6780 - val_categorical_accuracy: 0.6079 - val_loss: 1.5739
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - categorical_accuracy: 0.6315 - loss: 1.5177 - val_categorical_accuracy: 0.6689 - val_loss: 1.4385
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - categorical_accuracy: 0.6617 - loss: 1.4156 - val_categorical_accuracy: 0.6869 - val_loss: 1.3536
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - categorical_accuracy: 0.6840 - loss: 1.3436 - val_categorical_accuracy: 0.7079 - val_loss: 1.2937
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - categorical_accuracy: 0.7060 - loss: 1.2874 - val_categorical_accuracy: 0.7191 - val_loss: 1.2487
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - categorical_accuracy: 0.71

In [35]:
### Recompile so AdamW momenta are reset
learning_rate=5.0e-4 
weight_decay=1.0e-5 

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

text2text_calculator.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])

scheduler_patience = 3
stopper_patience = 12

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

### Fine-tune training
history_calc = text2text_calculator.fit(x=[X_train_calc, y_train_in_calc], y=y_train_target_calc, 
               epochs = 60,
               batch_size = 32,
               validation_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
               callbacks=[lr_scheduler, early_stopper],
               verbose=1)

Epoch 1/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - categorical_accuracy: 0.9962 - loss: 0.5929 - val_categorical_accuracy: 0.9985 - val_loss: 0.5846 - learning_rate: 5.0000e-04
Epoch 2/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - categorical_accuracy: 0.9978 - loss: 0.5877 - val_categorical_accuracy: 0.9940 - val_loss: 0.5944 - learning_rate: 5.0000e-04
Epoch 3/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - categorical_accuracy: 0.9967 - loss: 0.5917 - val_categorical_accuracy: 0.9989 - val_loss: 0.5817 - learning_rate: 5.0000e-04
Epoch 4/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - categorical_accuracy: 0.9969 - loss: 0.5902 - val_categorical_accuracy: 0.9934 - val_loss: 0.6012 - learning_rate: 5.0000e-04
Epoch 5/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - categorical_accuracy: 0.9972 - loss: 0.5898 - val_categorical_accuracy: 0.9990 - val_loss: 0.5803 - learning_rate: 5.0000e-04
Epoch 6/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - categorical_accuracy: 0.9983 - loss: 0.5

KeyboardInterrupt: 

#### Full model

In [69]:
# --- 1. Building full model ---
image2text = build_image2text(image2text_pretraining, text2text_calculator)
image2text_pretraining.trainable = False

# --- 2. Phase 1: Feature Alignment (Static Encoder) ---
learning_rate_static = 5e-4
weight_decay_static = 1e-4

optimizer_static = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate_static,
    weight_decay=weight_decay_static
)

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

image2text.compile(
    optimizer=optimizer_static, 
    loss=loss_fn,
    metrics=['categorical_accuracy']
)

# Note: 'y_train_expr' and 'y_val_expr' are the ground-truth math expressions
# Note: 'y_train_in' and 'y_val_in' are the ground-truth calculated results
validation_data = ([X_val, y_val_in_pt, y_val_in], y_val_target)

history_full_static = image2text.fit(
    x=[X_train, y_train_in_pt, y_train_in], 
    y=y_train_target, 
    epochs=15,
    batch_size=32,
    validation_data=validation_data)


Epoch 1/15
500/500 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - categorical_accuracy: 0.9637 - loss: 0.6968 - val_categorical_accuracy: 0.9079 - val_loss: 0.8601
Epoch 2/15
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - categorical_accuracy: 0.9652 - loss: 0.6920 - val_categorical_accuracy: 0.9082 - val_loss: 0.8619
Epoch 3/15
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - categorical_accuracy: 0.9634 - loss: 0.6955 - val_categorical_accuracy: 0.9084 - val_loss: 0.8584
Epoch 4/15
500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - categorical_accuracy: 0.9649 - loss: 0.6910 - val_categorical_accuracy: 0.9074 - val_loss: 0.8503
Epoch 5/15
500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - categorical_accuracy: 0.9652 - loss: 0.6910 - val_categorical_accuracy: 0.9100 - val_loss: 0.8507
Epoch 6/15
500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - categorical_accuracy: 0.9651 - loss: 0.6895 - val_categorical_accuracy: 0.9070 - val_loss: 0.8540
Epoch 7/15
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 22ms/step - categorical_accuracy

In [ ]:
# --- 3. Phase 2: End-to-End Fine-Tuning (Dynamic Encoder) ---
image2text_pretraining.trainable = True

learning_rate_dynamic = 1.0e-5
weight_decay_dynamic = 1e-2  # Increased to maintain regularization for the 97% encoder

optimizer_dynamic = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate_dynamic,
    weight_decay=weight_decay_dynamic
)

image2text.compile(
    optimizer=optimizer_dynamic, 
    loss=loss_fn,
    metrics=['categorical_accuracy']
)

# Callbacks
stopper_patience = 30
scheduler_patience = 10

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

# Final Fine-Tuning Fit
history_full_dynamic = image2text.fit(
    x=[X_train, y_train_in_pt, y_train_in], 
    y=y_train_target, 
    epochs=120,
    batch_size=32,
    validation_data=validation_data,
    callbacks=[lr_scheduler, early_stopper]
)

In [48]:
image2text.summary()

Model: "dual_teacher_forcing_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_sequence      │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expression_teacher… │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pretraining_model   │ (None, 6, 15)     │  1,254,991 │ image_sequence[0… │
│ (Functional)        │                   │            │ expression_teach… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ answer_teacher_for… │ (None, 4, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ calculator          │ (None, 4, 15)     │    696,591 │ pretraining_mode… │
│ (Functional)        │                   │            │ answer_teacher_f… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,854,364 (22.33 MB)

 Trainable params: 1,951,390 (7.44 MB)

 Non-trainable params: 192 (768.00 B)

 Optimizer params: 3,902,782 (14.89 MB)

## Inference loop

In [61]:
def run_full_dataset_inference_fixed_v2(X_data, y_true, full_model, batch_size=64):
    num_samples = X_data.shape[0]
    all_pred_indices = []
    
    encoder = full_model.get_layer('pretraining_model')
    calculator = full_model.get_layer('calculator')

    for i in range(0, num_samples, batch_size):
        X_batch = X_data[i:i + batch_size]
        curr_batch_size = X_batch.shape[0]

        # 1. Get Latents
        dummy_expr = np.zeros((curr_batch_size, 6, 15))
        expr_latents = encoder.predict([X_batch, dummy_expr], verbose=0)

        # 2. Autoregressive Loop
        res_seq = np.zeros((curr_batch_size, 4, 15))
        res_seq[:, 0, 1] = 1.0 # Set SOS at index 0

        for t in range(0, 3): # Process steps 0, 1, 2
            preds = calculator.predict([expr_latents, res_seq], verbose=0)
            
            # CRITICAL: We take the prediction FROM step 't' 
            # and place it INTO step 't+1'
            next_token_idx = np.argmax(preds[:, t, :], axis=-1)
            
            for j in range(curr_batch_size):
                if t + 1 < 4:
                    res_seq[j, t + 1, next_token_idx[j]] = 1.0
        
        all_pred_indices.append(np.argmax(res_seq, axis=-1))

    y_pred_idx = np.concatenate(all_pred_indices, axis=0)
    y_true_idx = np.argmax(y_true, axis=-1)

    # Compare ONLY the content tokens (Indices 1, 2, 3)
    # This ignores the SOS at index 0
    token_match = (y_pred_idx[:, 1:] == y_true_idx[:, 1:])
    math_acc = np.mean(np.all(token_match, axis=1))
    
    return math_acc, y_pred_idx, y_true_idx

math_acc, y_pred, y_true = run_full_dataset_inference_fixed_v2(X_val, y_val_target, image2text)
print(f"Mathematical Accuracy: {math_acc}")

Mathematical Accuracy: 0.0


In [65]:
import numpy as np

# 1. Convert everything to integer indices first
# If y_pred_final is already the result of argmax, skip this. 
# But based on your output, it is currently one-hot.
y_pred_idx = np.argmax(y_pred_final, axis=-1) if y_pred_final.ndim == 3 else y_pred_final
y_true_idx = np.argmax(y_val_target, axis=-1)

# 2. Align the comparison
# Predicted: [SOS, Digit1, Digit2, Digit3] -> We want [Digit1, Digit2, Digit3]
# True:      [Digit1, Digit2, Digit3, EOS] -> We want [Digit1, Digit2, Digit3]

# We take indices 1, 2, 3 from prediction
# We take indices 0, 1, 2 from truth
preds_to_compare = y_pred_idx[:, 1:4] 
truth_to_compare = y_true_idx[:, 0:3]

# 3. Calculate Accuracy
token_match = (preds_to_compare == truth_to_compare)
token_acc = np.mean(token_match)
math_acc = np.mean(np.all(token_match, axis=1))

print(f"--- Corrected Metrics ---")
print(f"Token Accuracy: {token_acc:.4f}")
print(f"Mathematical Accuracy (Exact Match): {math_acc:.4f}")

# Visual verification
for i in range(10):
    print(f"Sample {i} | Pred: {preds_to_compare[i]} | True: {truth_to_compare[i]}")

--- Corrected Metrics ---
Token Accuracy: 0.6478
Mathematical Accuracy (Exact Match): 0.3078
Sample 0 | Pred: [11  3  6] | True: [11  3  6]
Sample 1 | Pred: [11  3  8] | True: [11  4  8]
Sample 2 | Pred: [1 1 0] | True: [1 1 0]
Sample 3 | Pred: [ 9 12 12] | True: [ 9 12 12]
Sample 4 | Pred: [ 5 12 12] | True: [ 1 12 12]
Sample 5 | Pred: [12 12 12] | True: [11  1  5]
Sample 6 | Pred: [11  6  6] | True: [11  6  9]
Sample 7 | Pred: [1 5 7] | True: [1 5 6]
Sample 8 | Pred: [11  5  5] | True: [11  6  5]
Sample 9 | Pred: [ 5 12 12] | True: [ 1  5 12]
